# <font color="#418FDE" size="6.5" uppercase>**Neural Network Optimization and Training**</font>

>Last update: 20260327.
    
By the end of this Lecture, you will be able to:
- Formulate neural network training as an optimization problem involving loss functions, gradients, and parameter updates. 
- Implement gradient-based training methods, including gradient descent, stochastic gradient descent, and backpropagation. 
- Evaluate the effects of optimization choices such as learning rate, momentum, batch size, and Adam on neural network. 


## **1. Neural Network Optimization**

### **1.1. Optimization Problem Formulation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_05/Lecture_B/image_01_01.jpg?v=1774623216" width="250">



>* Training adjusts weights and biases for tasks.
>* Loss on examples guides parameter improvement.

>* Training searches a high-dimensional loss landscape.
>* Gradients guide parameter updates toward lower loss.

>* Optimization defines learning as minimizing error.
>* Loss and update choices shape performance.



In [1]:
#@title Python Code - Optimization Problem Formulation

# Neural networks learn by minimizing prediction error.
# Concrete strength offers a civil engineering example.
# This script frames training as optimization.

# Download the concrete strength dataset.
!wget -q -O ./Concrete_Data.xls https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls

# Import only beginner friendly libraries.
import numpy as np
import pandas as pd

# Set a seed for repeatability.
np.random.seed(7)

# Load the spreadsheet into pandas.
df = pd.read_excel("./Concrete_Data.xls")

# Keep a small subset for speed.
df = df.iloc[:300].copy()

# Separate inputs and target values.
X = df.iloc[:, :-1].to_numpy(dtype=float)
y = df.iloc[:, -1].to_numpy(dtype=float).reshape(-1, 1)

# Check that data shapes look valid.
assert X.ndim == 2 and y.ndim == 2
assert X.shape[0] == y.shape[0]

# Standardize inputs for stable optimization.
X_mean = X.mean(axis=0, keepdims=True)
X_std = X.std(axis=0, keepdims=True) + 1e-8
X = (X - X_mean) / X_std

# Standardize targets for easier learning.
y_mean = y.mean(axis=0, keepdims=True)
y_std = y.std(axis=0, keepdims=True) + 1e-8
y_scaled = (y - y_mean) / y_std

# Build a tiny one hidden layer network.
input_size = X.shape[1]
hidden_size = 5
W1 = 0.1 * np.random.randn(input_size, hidden_size)
b1 = np.zeros((1, hidden_size))

# Initialize second layer parameters.
W2 = 0.1 * np.random.randn(hidden_size, 1)
b2 = np.zeros((1, 1))

# Define the forward prediction step.
def forward_pass(X_data, W1_data, b1_data, W2_data, b2_data):
    Z1 = X_data @ W1_data + b1_data

    A1 = np.tanh(Z1)
    Y_hat = A1 @ W2_data + b2_data
    return Z1, A1, Y_hat

# Define mean squared error loss.
def mse_loss(y_true, y_pred):
    diff = y_pred - y_true

    return np.mean(diff ** 2)

# Compute the starting loss value.
Z1, A1, y_pred = forward_pass(X, W1, b1, W2, b2)
initial_loss = mse_loss(y_scaled, y_pred)

# Choose optimization settings.
learning_rate = 0.05
epochs = 200
n = X.shape[0]

# Run gradient based optimization.
for epoch in range(epochs):
    Z1, A1, y_pred = forward_pass(X, W1, b1, W2, b2)
    dY = (2.0 / n) * (y_pred - y_scaled)

    dW2 = A1.T @ dY
    db2 = np.sum(dY, axis=0, keepdims=True)
    dA1 = dY @ W2.T

    dZ1 = dA1 * (1.0 - np.tanh(Z1) ** 2)
    dW1 = X.T @ dZ1
    db1 = np.sum(dZ1, axis=0, keepdims=True)

    W2 = W2 - learning_rate * dW2
    b2 = b2 - learning_rate * db2
    W1 = W1 - learning_rate * dW1
    b1 = b1 - learning_rate * db1

# Measure the final optimized loss.
Z1, A1, y_pred = forward_pass(X, W1, b1, W2, b2)
final_loss = mse_loss(y_scaled, y_pred)

# Convert predictions back to MPa units.
y_pred_mpa = y_pred * y_std + y_mean
sample_error = float(y_pred_mpa[0, 0] - y[0, 0])

# Print a compact optimization summary.
print("Dataset rows used:", X.shape[0])
print("Input features:", X.shape[1])
print("Target: concrete compressive strength")
print("Objective: minimize mean squared error")
print("Initial loss:", round(float(initial_loss), 4))
print("Final loss:", round(float(final_loss), 4))
print("Learning rate:", learning_rate)
print("Example true strength MPa:", round(float(y[0, 0]), 2))
print("Example predicted MPa:", round(float(y_pred_mpa[0, 0]), 2))
print("Example prediction error MPa:", round(sample_error, 2))



Dataset rows used: 300
Input features: 8
Target: concrete compressive strength
Objective: minimize mean squared error
Initial loss: 0.994
Final loss: 0.2675
Learning rate: 0.05
Example true strength MPa: 79.99
Example predicted MPa: 59.76
Example prediction error MPa: -20.23


### **1.2. Backpropagation and Parameters**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_05/Lecture_B/image_01_02.jpg?v=1774623256" width="250">



>* Backpropagation links prediction errors to parameters.
>* It shows how weights should change.

>* Backpropagation traces each parameter's effect on error.
>* Gradients guide proportional updates across all layers.

>* Weights and biases define network behavior.
>* Backpropagation tunes parameters to reduce error.



In [2]:
#@title Python Code - Backpropagation and Parameters

# Backpropagation links errors to parameter updates.
# Concrete strength offers a civil engineering example.
# This notebook builds one tiny network.

# Download the concrete strength dataset.
!wget -q -O ./Concrete_Data.xls https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls

# Import only beginner friendly libraries.
import numpy as np
import pandas as pd

# Fix randomness for repeatable results.
np.random.seed(7)

# Load the spreadsheet into pandas.
df = pd.read_excel("./Concrete_Data.xls")

# Keep a small subset for speed.
df = df.iloc[:240].copy()

# Separate inputs and target values.
X_df = df.iloc[:, :-1]
y_df = df.iloc[:, -1:]

# Convert tables into numpy arrays.
X = X_df.to_numpy(dtype=float)
y = y_df.to_numpy(dtype=float)

# Check basic shapes before training.
assert X.shape[0] == y.shape[0]
assert X.shape[1] == 8

# Standardize inputs using training statistics.
X_mean = X.mean(axis=0, keepdims=True)
X_std = X.std(axis=0, keepdims=True) + 1e-8
X = (X - X_mean) / X_std

# Standardize target for stable learning.
y_mean = y.mean(axis=0, keepdims=True)
y_std = y.std(axis=0, keepdims=True) + 1e-8
y = (y - y_mean) / y_std

# Split into train and test parts.
train_size = 200
X_train = X[:train_size]
y_train = y[:train_size]
X_test = X[train_size:]
y_test = y[train_size:]

# Create a tiny neural network.
input_size = X_train.shape[1]
hidden_size = 5
output_size = 1

# Initialize weights and biases.
W1 = 0.1 * np.random.randn(input_size, hidden_size)
b1 = np.zeros((1, hidden_size))
W2 = 0.1 * np.random.randn(hidden_size, output_size)
b2 = np.zeros((1, output_size))

# Define a simple activation function.
def relu(z):
    return np.maximum(0.0, z)

# Define the activation derivative.
def relu_grad(z):
    return (z > 0).astype(float)

# Run one forward pass.
def forward_pass(X_batch, W1, b1, W2, b2):
    z1 = X_batch @ W1 + b1
    a1 = relu(z1)
    z2 = a1 @ W2 + b2
    return z1, a1, z2

# Compute mean squared error loss.
def mse_loss(y_true, y_pred):
    diff = y_pred - y_true
    return np.mean(diff * diff)

# Train with mini batch gradient descent.
learning_rate = 0.03
epochs = 120
batch_size = 20
first_grad_w1 = None
first_grad_b1 = None
first_grad_w2 = None
first_grad_b2 = None

# Loop through several training epochs.
for epoch in range(epochs):

    # Shuffle training rows each epoch.
    order = np.random.permutation(X_train.shape[0])
    X_train = X_train[order]
    y_train = y_train[order]

    # Visit one mini batch at a time.
    for start in range(0, X_train.shape[0], batch_size):
        end = start + batch_size
        X_batch = X_train[start:end]
        y_batch = y_train[start:end]

        # Compute predictions from current parameters.
        z1, a1, y_pred = forward_pass(
            X_batch, W1, b1, W2, b2
        )

        # Backpropagate loss through the network.
        m = X_batch.shape[0]
        dloss_dy = (2.0 / m) * (y_pred - y_batch)
        grad_W2 = a1.T @ dloss_dy
        grad_b2 = np.sum(dloss_dy, axis=0, keepdims=True)

        # Continue gradients into hidden layer.
        hidden_error = dloss_dy @ W2.T
        hidden_error = hidden_error * relu_grad(z1)
        grad_W1 = X_batch.T @ hidden_error
        grad_b1 = np.sum(hidden_error, axis=0, keepdims=True)

        # Save one gradient example once.
        if first_grad_w1 is None:
            first_grad_w1 = grad_W1.copy()
            first_grad_b1 = grad_b1.copy()
            first_grad_w2 = grad_W2.copy()
            first_grad_b2 = grad_b2.copy()

        # Update weights and biases.
        W2 = W2 - learning_rate * grad_W2
        b2 = b2 - learning_rate * grad_b2
        W1 = W1 - learning_rate * grad_W1
        b1 = b1 - learning_rate * grad_b1

# Measure final train and test losses.
_, _, train_pred = forward_pass(X_train, W1, b1, W2, b2)
_, _, test_pred = forward_pass(X_test, W1, b1, W2, b2)
train_loss = mse_loss(y_train, train_pred)
test_loss = mse_loss(y_test, test_pred)

# Convert one prediction back to MPa.
one_pred = test_pred[0:1] * y_std + y_mean
one_true = y_test[0:1] * y_std + y_mean

# Summarize how backpropagation used parameters.
print("Rows used:", X.shape[0], "Features:", X.shape[1])
print("W1 shape:", W1.shape, "b1 shape:", b1.shape)
print("W2 shape:", W2.shape, "b2 shape:", b2.shape)
print("Initial grad W1 mean:", round(float(first_grad_w1.mean()), 4))
print("Initial grad b1 mean:", round(float(first_grad_b1.mean()), 4))
print("Initial grad W2 mean:", round(float(first_grad_w2.mean()), 4))
print("Initial grad b2 mean:", round(float(first_grad_b2.mean()), 4))
print("Final train loss:", round(float(train_loss), 4))
print("Final test loss:", round(float(test_loss), 4))
print("Example true MPa:", round(float(one_true[0, 0]), 2))
print("Example predicted MPa:", round(float(one_pred[0, 0]), 2))



Rows used: 240 Features: 8
W1 shape: (8, 5) b1 shape: (1, 5)
W2 shape: (5, 1) b2 shape: (1, 1)
Initial grad W1 mean: 0.0013
Initial grad b1 mean: -0.0184
Initial grad W2 mean: 0.0513
Initial grad b2 mean: 0.828
Final train loss: 0.1635
Final test loss: 0.2511
Example true MPa: 21.06
Example predicted MPa: 17.47


### **1.3. Loss Function Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_05/Lecture_B/image_01_03.jpg?v=1774623302" width="250">



>* Loss measures prediction error during training.
>* Minimizing loss guides learning toward correct outputs.

>* Loss choice depends on task output type.
>* It should reflect real application priorities.

>* Loss links errors to parameter updates.
>* Loss shape affects training stability and progress.



In [3]:
#@title Python Code - Loss Function Basics

# Loss functions turn errors into numbers.
# This example uses air quality data.
# We compare regression and classification losses.

# Install lines are unnecessary here.

import numpy as np
import pandas as pd
from zipfile import ZipFile

# Set a seed for repeatability.
np.random.seed(7)

# Download the dataset file.
!wget -q -O ./AirQualityUCI.zip https://archive.ics.uci.edu/ml/machine-learning-databases/00360/AirQualityUCI.zip

# Extract the compressed dataset.
with ZipFile("./AirQualityUCI.zip", "r") as zip_file:
    zip_file.extractall("./air_quality_data")

# Read the main csv file.
file_path = "./air_quality_data/AirQualityUCI.csv"
df = pd.read_csv(file_path, sep=";", decimal=",")

# Remove empty trailing columns.
df = df.dropna(axis=1, how="all")

# Replace missing markers safely.
df = df.replace(-200, np.nan)

# Keep useful numeric columns only.
cols = ["CO(GT)", "PT08.S1(CO)", "T", "RH"]
df = df[cols].dropna().copy()

# Use a small clean sample.
df = df.head(200).reset_index(drop=True)

# Build a simple regression target.
y_true_reg = df["CO(GT)"].to_numpy(dtype=float)

# Build one simple model output.
x_signal = df["PT08.S1(CO)"].to_numpy(dtype=float)
y_pred_reg = 0.002 * x_signal + 0.2

# Compute mean squared error.
reg_errors = y_true_reg - y_pred_reg
mse = np.mean(reg_errors ** 2)

# Build a binary target example.
median_co = float(np.median(y_true_reg))
y_true_bin = (y_true_reg > median_co).astype(float)

# Create simple probability outputs.
temp = df["T"].to_numpy(dtype=float)
humid = df["RH"].to_numpy(dtype=float)
logits = 0.25 * (temp - temp.mean()) - 0.03 * (humid - humid.mean())
y_pred_prob = 1.0 / (1.0 + np.exp(-logits))

# Clip probabilities for stability.
eps = 1e-9
y_pred_prob = np.clip(y_pred_prob, eps, 1.0 - eps)

# Compute binary cross entropy.
bce_terms = (
    y_true_bin * np.log(y_pred_prob)
    + (1.0 - y_true_bin) * np.log(1.0 - y_pred_prob)
)
bce = -np.mean(bce_terms)

# Show a few example predictions.
example = df[["CO(GT)", "PT08.S1(CO)", "T", "RH"]].head(3).copy()

# Print short teaching results.
print("Rows used:", len(df))
print("Regression target: CO(GT)")
print("Classification threshold:", round(median_co, 3))
print("Mean squared error:", round(float(mse), 4))
print("Binary cross-entropy:", round(float(bce), 4))
print("First regression predictions:", np.round(y_pred_reg[:3], 3))
print("First class probabilities:", np.round(y_pred_prob[:3], 3))
print("First true class labels:", y_true_bin[:3].astype(int))



Rows used: 200
Regression target: CO(GT)
Classification threshold: 2.5
Mean squared error: 1.3277
Binary cross-entropy: 0.681
First regression predictions: [2.92  2.784 3.004]
First class probabilities: [0.402 0.393 0.274]
First true class labels: [1 0 0]


## **2. Gradient Based Training**

### **2.1. Learning Rate Momentum**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_05/Lecture_B/image_02_01.jpg?v=1774623338" width="250">



>* Learning rate sets update step size.
>* Momentum carries direction, reducing zigzagging.

>* Momentum smooths noisy gradient updates.
>* It speeds stable learning across batches.

>* Balance learning rate and momentum carefully.
>* Monitor training for speed and stability.



In [4]:
#@title Python Code - Learning Rate Momentum

# Tiny network learns concrete strength trends.
# Learning rate changes update step size.
# Momentum smooths noisy gradient directions.

# Download the concrete dataset file.
!wget -q -O ./Concrete_Data.xls https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls

# Import only allowed teaching libraries.
import numpy as np
import pandas as pd

# Fix randomness for repeatable results.
np.random.seed(7)

# Load the spreadsheet into pandas.
df = pd.read_excel("./Concrete_Data.xls")

# Keep a small subset for speed.
df = df.sample(n=240, random_state=7)

# Split inputs and target values.
X = df.iloc[:, :-1].to_numpy(dtype=float)
y = df.iloc[:, -1].to_numpy(dtype=float).reshape(-1, 1)

# Check basic shapes before training.
assert X.shape[0] == y.shape[0]
assert X.shape[1] == 8

# Standardize features and target values.
X_mean = X.mean(axis=0, keepdims=True)
X_std = X.std(axis=0, keepdims=True) + 1e-8
y_mean = y.mean(axis=0, keepdims=True)
y_std = y.std(axis=0, keepdims=True) + 1e-8

# Apply scaling for stable gradients.
X = (X - X_mean) / X_std
y = (y - y_mean) / y_std

# Build a tiny train test split.
X_train = X[:200]
y_train = y[:200]
X_test = X[200:]
y_test = y[200:]

# Define a tiny neural network trainer.
def train_model(X_data, y_data, lr, momentum, epochs):

    # Create small random starting weights.
    W1 = 0.1 * np.random.randn(8, 6)
    b1 = np.zeros((1, 6))
    W2 = 0.1 * np.random.randn(6, 1)
    b2 = np.zeros((1, 1))

    # Start velocity terms at zero.
    vW1 = np.zeros_like(W1)
    vb1 = np.zeros_like(b1)
    vW2 = np.zeros_like(W2)
    vb2 = np.zeros_like(b2)

    # Store a short loss history.
    losses = []
    n = X_data.shape[0]

    # Run simple full batch training.
    for epoch in range(epochs):

        # Compute forward pass values.
        Z1 = X_data @ W1 + b1
        A1 = np.tanh(Z1)
        y_hat = A1 @ W2 + b2

        # Measure mean squared error.
        error = y_hat - y_data
        loss = np.mean(error ** 2)
        losses.append(loss)

        # Compute backpropagation gradients.
        dy = (2.0 / n) * error
        dW2 = A1.T @ dy
        db2 = np.sum(dy, axis=0, keepdims=True)
        dA1 = dy @ W2.T

        # Backpropagate through tanh layer.
        dZ1 = dA1 * (1.0 - A1 ** 2)
        dW1 = X_data.T @ dZ1
        db1 = np.sum(dZ1, axis=0, keepdims=True)

        # Update velocities using momentum.
        vW2 = momentum * vW2 - lr * dW2
        vb2 = momentum * vb2 - lr * db2
        vW1 = momentum * vW1 - lr * dW1
        vb1 = momentum * vb1 - lr * db1

        # Apply parameter updates now.
        W2 = W2 + vW2
        b2 = b2 + vb2
        W1 = W1 + vW1
        b1 = b1 + vb1

    # Return learned values and losses.
    return W1, b1, W2, b2, losses

# Define a prediction helper function.
def predict_values(X_data, W1, b1, W2, b2):

    # Run one forward pass.
    hidden = np.tanh(X_data @ W1 + b1)
    output = hidden @ W2 + b2
    return output

# Train three settings for comparison.
slow = train_model(X_train, y_train, 0.01, 0.0, 120)
fast = train_model(X_train, y_train, 0.60, 0.0, 120)
momo = train_model(X_train, y_train, 0.05, 0.9, 120)

# Compute test losses for each setting.
slow_pred = predict_values(X_test, slow[0], slow[1], slow[2], slow[3])
fast_pred = predict_values(X_test, fast[0], fast[1], fast[2], fast[3])
momo_pred = predict_values(X_test, momo[0], momo[1], momo[2], momo[3])

# Measure mean squared error values.
slow_test = np.mean((slow_pred - y_test) ** 2)
fast_test = np.mean((fast_pred - y_test) ** 2)
momo_test = np.mean((momo_pred - y_test) ** 2)

# Summarize convergence and oscillation behavior.
print("Dataset rows used:", X.shape[0])
print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Small lr final train loss:", round(slow[4][-1], 4))
print("Large lr final train loss:", round(fast[4][-1], 4))
print("Momentum final train loss:", round(momo[4][-1], 4))
print("Small lr test loss:", round(slow_test, 4))
print("Large lr test loss:", round(fast_test, 4))
print("Momentum test loss:", round(momo_test, 4))
print("Large lr early losses:", np.round(fast[4][:5], 3))
print("Momentum early losses:", np.round(momo[4][:5], 3))
print("Idea: momentum often reduces zigzagging.")



Dataset rows used: 240
Train shape: (200, 8) Test shape: (40, 8)
Small lr final train loss: 0.5964
Large lr final train loss: 0.191
Momentum final train loss: 0.1223
Small lr test loss: 0.6441
Large lr test loss: 0.2811
Momentum test loss: 0.2113
Large lr early losses: [1.002 0.893 0.717 0.458 0.342]
Momentum early losses: [1.087 1.068 1.037 1.003 0.972]
Idea: momentum often reduces zigzagging.


### **2.2. Gradient Descent Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_05/Lecture_B/image_02_02.jpg?v=1774623377" width="250">



>* Loss measures prediction error during training.
>* Gradients guide parameter updates toward lower loss.

>* Gradients coordinate updates across many parameters.
>* Repeated tuning helps networks learn useful patterns.

>* Training improves through repeated gradient updates.
>* Gradients guide progress despite uneven loss.



In [5]:
#@title Python Code - Gradient Descent Basics

# Gradient descent trains networks through repeated updates.
# This example uses concrete strength data.
# Full batch learning shows basic optimization.

# Download the concrete dataset file.
!wget -q -O ./Concrete_Data.xls https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls

# Import beginner friendly data tools.
import numpy as np
import pandas as pd

# Set a deterministic random seed.
np.random.seed(7)

# Load the spreadsheet into pandas.
df = pd.read_excel("./Concrete_Data.xls")

# Keep a small subset for speed.
df = df.iloc[:200].copy()

# Split inputs and target values.
X = df.iloc[:, :-1].to_numpy(dtype=float)
y = df.iloc[:, -1].to_numpy(dtype=float).reshape(-1, 1)

# Check that data shapes look valid.
assert X.shape[0] == y.shape[0]
assert X.shape[1] > 0 and y.shape[1] == 1

# Standardize inputs using training statistics.
X_mean = X.mean(axis=0, keepdims=True)
X_std = X.std(axis=0, keepdims=True) + 1e-8
X = (X - X_mean) / X_std

# Standardize target for stable learning.
y_mean = y.mean(axis=0, keepdims=True)
y_std = y.std(axis=0, keepdims=True) + 1e-8
y = (y - y_mean) / y_std

# Define a tiny neural network size.
input_size = X.shape[1]
hidden_size = 6
output_size = 1

# Initialize weights with small values.
W1 = 0.1 * np.random.randn(input_size, hidden_size)
b1 = np.zeros((1, hidden_size))
W2 = 0.1 * np.random.randn(hidden_size, output_size)
b2 = np.zeros((1, output_size))

# Choose learning settings for training.
learning_rate = 0.05
iterations = 8
n = X.shape[0]

# Explain the training example briefly.
print("Concrete samples:", n, "features:", input_size)
print("Training with full batch gradient descent")

# Run several gradient descent steps.
for step in range(iterations):

    # Compute hidden layer activations.
    Z1 = X @ W1 + b1
    A1 = np.maximum(0, Z1)

    # Compute output predictions.
    Y_hat = A1 @ W2 + b2
    error = Y_hat - y

    # Measure mean squared error loss.
    loss = np.mean(error ** 2)
    print("Iteration", step + 1, "loss", round(float(loss), 4))

    # Backpropagate output layer gradients.
    dY = (2.0 / n) * error
    dW2 = A1.T @ dY
    db2 = np.sum(dY, axis=0, keepdims=True)

    # Backpropagate hidden layer gradients.
    dA1 = dY @ W2.T
    dZ1 = dA1 * (Z1 > 0)
    dW1 = X.T @ dZ1
    db1 = np.sum(dZ1, axis=0, keepdims=True)

    # Update parameters using gradients.
    W2 = W2 - learning_rate * dW2
    b2 = b2 - learning_rate * db2
    W1 = W1 - learning_rate * dW1
    b1 = b1 - learning_rate * db1

print("Final standardized prediction loss shown above")



Concrete samples: 200 features: 8
Training with full batch gradient descent
Iteration 1 loss 1.0019
Iteration 2 loss 0.9977
Iteration 3 loss 0.9938
Iteration 4 loss 0.99
Iteration 5 loss 0.9863
Iteration 6 loss 0.9825
Iteration 7 loss 0.9788
Iteration 8 loss 0.9751
Final standardized prediction loss shown above


### **2.3. Mini Batch Training**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_05/Lecture_B/image_02_03.jpg?v=1774623409" width="250">



>* Mini batches balance speed and stable learning.
>* Small groups enable efficient repeated updates.

>* Batch size changes gradient noise and smoothness.
>* Mini batches balance efficiency, stability, and learning.

>* Shuffle data; update after each mini batch.
>* Small repeated feedback improves learning efficiently.



In [6]:
#@title Python Code - Mini Batch Training

# Mini batch training uses small shuffled groups.
# This example uses air quality data.
# We compare SGD and mini batches.

# Download the dataset for this lesson.
!wget -q -O ./AirQualityUCI.zip https://archive.ics.uci.edu/ml/machine-learning-databases/00360/AirQualityUCI.zip

# Unzip the downloaded dataset file.
!unzip -oq ./AirQualityUCI.zip

# Import only beginner friendly libraries.
import numpy as np
import pandas as pd

# Set a seed for repeatable results.
np.random.seed(7)

# Load the semicolon separated dataset.
df = pd.read_csv(
    "AirQualityUCI.csv", sep=";", decimal="," 
)

# Remove empty columns from the file.
df = df.dropna(axis=1, how="all")

# Replace missing value markers safely.
df = df.replace(-200, np.nan)

# Keep a few useful numeric columns.
cols = ["CO(GT)", "PT08.S1(CO)", "T", "RH", "AH"]
df = df[cols].dropna().copy()

# Limit rows for a fast classroom run.
df = df.iloc[:1000].copy()

# Build features and one target.
X_df = df[["PT08.S1(CO)", "T", "RH", "AH"]]
y_df = df[["CO(GT)"]]

# Convert tables into numpy arrays.
X = X_df.to_numpy(dtype=float)
y = y_df.to_numpy(dtype=float)

# Check that data sizes are valid.
assert X.shape[0] > 100 and X.shape[1] == 4

# Standardize features for stable training.
X_mean = X.mean(axis=0, keepdims=True)
X_std = X.std(axis=0, keepdims=True) + 1e-8
X = (X - X_mean) / X_std

# Standardize target for easier optimization.
y_mean = y.mean(axis=0, keepdims=True)
y_std = y.std(axis=0, keepdims=True) + 1e-8
y = (y - y_mean) / y_std

# Add a bias column of ones.
ones = np.ones((X.shape[0], 1))
X = np.hstack((ones, X))

# Split into train and test parts.
split = int(0.8 * X.shape[0])
X_train = X[:split]
y_train = y[:split]
X_test = X[split:]
y_test = y[split:]

# Define mean squared error loss.
def mse_loss(X_data, y_data, w):
    pred = X_data @ w
    err = pred - y_data
    return np.mean(err * err)

# Define one gradient step helper.
def batch_gradient(X_data, y_data, w):
    pred = X_data @ w
    err = pred - y_data
    grad = (2.0 / X_data.shape[0]) * (X_data.T @ err)
    return grad

# Train using shuffled mini batches.
def train_model(X_data, y_data, epochs, lr, batch_size):
    w = np.zeros((X_data.shape[1], 1))
    n = X_data.shape[0]
    history = []

    # Repeat over several training epochs.
    for epoch in range(epochs):
        order = np.random.permutation(n)
        X_shuf = X_data[order]
        y_shuf = y_data[order]

        # Process one small batch at a time.
        for start in range(0, n, batch_size):
            end = start + batch_size
            X_batch = X_shuf[start:end]
            y_batch = y_shuf[start:end]

            # Update weights from batch gradients.
            grad = batch_gradient(X_batch, y_batch, w)
            w = w - lr * grad

        # Save one loss value each epoch.
        history.append(mse_loss(X_data, y_data, w))

    return w, history

# Use one sample for stochastic descent.
sgd_size = 1

# Use a small group for mini batches.
mini_size = 32

# Count batches seen in one epoch.
sgd_batches = int(np.ceil(X_train.shape[0] / sgd_size))
mini_batches = int(np.ceil(X_train.shape[0] / mini_size))

# Train with stochastic gradient descent.
w_sgd, h_sgd = train_model(
    X_train, y_train, 12, 0.01, sgd_size
)

# Train with mini batch gradient descent.
w_mini, h_mini = train_model(
    X_train, y_train, 12, 0.01, mini_size
)

# Evaluate both trained models.
sgd_test = mse_loss(X_test, y_test, w_sgd)
mini_test = mse_loss(X_test, y_test, w_mini)

# Print a short teaching summary.
print("Rows used:", X.shape[0], " Train rows:", X_train.shape[0])
print("Features plus bias:", X_train.shape[1])
print("SGD batch size:", sgd_size, " Batches:", sgd_batches)
print("Mini batch size:", mini_size, " Batches:", mini_batches)
print("Shuffling changes sample order each epoch.")
print("Mini batches need less memory than full data.")
print("SGD final train loss:", round(h_sgd[-1], 4))
print("Mini final train loss:", round(h_mini[-1], 4))
print("SGD test loss:", round(sgd_test, 4))
print("Mini test loss:", round(mini_test, 4))



Rows used: 1000  Train rows: 800
Features plus bias: 5
SGD batch size: 1  Batches: 800
Mini batch size: 32  Batches: 25
Shuffling changes sample order each epoch.
Mini batches need less memory than full data.
SGD final train loss: 0.1274
Mini final train loss: 0.125
SGD test loss: 0.1404
Mini test loss: 0.1253


## **3. Adaptive Optimization Methods**

### **3.1. Parameter Updates**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_05/Lecture_B/image_03_01.jpg?v=1774623454" width="250">



>* Updates adjust weights using loss gradients.
>* Step size controls training speed and stability.

>* Momentum smooths updates using past gradients.
>* Batch size trades noise for stability.

>* Adaptive methods tailor step sizes by parameter.
>* Faster training may trade off generalization.



In [7]:
#@title Python Code - Parameter Updates

# Concrete strength updates show learning mechanics.
# Small neural network uses one data sample.
# Parameter updates are printed step by step.

import numpy as np
import pandas as pd

# Download the concrete dataset file.
!wget -q -O ./Concrete_Data.xls https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls

# Set a deterministic random seed.
np.random.seed(7)

# Load the spreadsheet into pandas.
df = pd.read_excel("./Concrete_Data.xls")

# Keep only numeric values safely.
df = df.select_dtypes(include=[np.number]).dropna()

# Check dataset size before using it.
if df.shape[0] < 1 or df.shape[1] < 9:
    raise ValueError("Dataset shape is smaller than expected.")

# Use one concrete mix sample.
row = df.iloc[0]

# Select three input features.
feature_names = list(df.columns[:3])

# Select the target strength column.
target_name = df.columns[-1]

# Build input and target arrays.
x_raw = row[feature_names].to_numpy(dtype=float).reshape(1, 3)
y_raw = np.array([[float(row[target_name])]])

# Scale values for stable learning.
x = x_raw / x_raw.max()
y = y_raw / 100.0

# Define a tiny neural network.
W1 = np.array([[0.10, -0.20], [0.05, 0.10], [-0.10, 0.15]])
b1 = np.array([[0.01, -0.02]])

# Define output layer parameters.
W2 = np.array([[0.20], [-0.10]])
b2 = np.array([[0.03]])

# Choose a simple learning rate.
learning_rate = 0.10

# Compute hidden layer pre-activation.
z1 = x @ W1 + b1

# Apply ReLU activation function.
a1 = np.maximum(0.0, z1)

# Compute output prediction value.
y_hat = a1 @ W2 + b2

# Compute mean squared error.
loss = np.mean((y_hat - y) ** 2)

# Differentiate loss at output.
dL_dyhat = 2.0 * (y_hat - y)

# Compute output gradients.
dW2 = a1.T @ dL_dyhat
db2 = dL_dyhat.copy()

# Backpropagate into hidden layer.
da1 = dL_dyhat @ W2.T
relu_grad = (z1 > 0).astype(float)

# Compute hidden layer gradients.
dz1 = da1 * relu_grad
dW1 = x.T @ dz1

# Compute hidden bias gradient.
db1 = dz1.copy()

# Save old parameters for comparison.
old_W1 = W1.copy()
old_b1 = b1.copy()
old_W2 = W2.copy()
old_b2 = b2.copy()

# Update parameters using gradient descent.
W1 = W1 - learning_rate * dW1
b1 = b1 - learning_rate * db1
W2 = W2 - learning_rate * dW2
b2 = b2 - learning_rate * db2

# Compute new prediction after update.
new_z1 = x @ W1 + b1
new_a1 = np.maximum(0.0, new_z1)
new_y_hat = new_a1 @ W2 + b2

# Compute new loss after update.
new_loss = np.mean((new_y_hat - y) ** 2)

# Print a compact teaching summary.
print("Features used:", feature_names)
print("Target used:", target_name)
print("Scaled input:", np.round(x, 3))
print("Scaled target and prediction:", np.round(y, 3), np.round(y_hat, 3))
print("Initial loss:", round(float(loss), 6))
print("dW2 and db2:", np.round(dW2.ravel(), 6), np.round(db2.ravel(), 6))
print("dW1 first column:", np.round(dW1[:, 0], 6))
print("Old W2 and new W2:", np.round(old_W2.ravel(), 6), np.round(W2.ravel(), 6))
print("Old b2 and new b2:", np.round(old_b2.ravel(), 6), np.round(b2.ravel(), 6))
print("Old W1[0] and new W1[0]:", np.round(old_W1[0], 6), np.round(W1[0], 6))
print("New prediction and loss:", np.round(new_y_hat, 3), round(float(new_loss), 6))



Features used: ['Cement (component 1)(kg in a m^3 mixture)', 'Blast Furnace Slag (component 2)(kg in a m^3 mixture)', 'Fly Ash (component 3)(kg in a m^3 mixture)']
Target used: Concrete compressive strength(MPa, megapascals) 
Scaled input: [[1. 0. 0.]]
Scaled target and prediction: [[0.8]] [[0.052]]
Initial loss: 0.559296
dW2 and db2: [-0.164529  0.      ] [-1.495722]
dW1 first column: [-0.299144  0.        0.      ]
Old W2 and new W2: [ 0.2 -0.1] [ 0.216453 -0.1     ]
Old b2 and new b2: [0.03] [0.179572]
Old W1[0] and new W1[0]: [ 0.1 -0.2] [ 0.129914 -0.2     ]
New prediction and loss: [[0.216]] 0.340506


### **3.2. Adam Optimization**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_05/Lecture_B/image_03_02.jpg?v=1774623490" width="250">



>* Adam combines momentum with adaptive step sizes.
>* It speeds training with less manual tuning.

>* Adam smooths noisy gradients with adaptive updates.
>* Still needs tuning and sound modeling.

>* Adam learns quickly, useful for experimentation.
>* Compare validation and generalization against simpler methods.



In [8]:
#@title Python Code - Adam Optimization

# Adam helps stabilize noisy training updates.
# Concrete strength offers a civil example.
# We compare gradient descent and Adam.

# !pip install xlrd.

# Download the concrete strength dataset.
!wget -q -O ./Concrete_Data.xls https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls

# Import only beginner friendly libraries.
import numpy as np
import pandas as pd

# Set a deterministic random seed.
np.random.seed(7)

# Load the spreadsheet into pandas.
df = pd.read_excel("./Concrete_Data.xls")

# Keep a small subset for speed.
df = df.sample(n=300, random_state=7)

# Convert data into numeric arrays.
X = df.iloc[:, :-1].to_numpy(dtype=float)
y = df.iloc[:, -1].to_numpy(dtype=float).reshape(-1, 1)

# Check that shapes look valid.
assert X.shape[0] == y.shape[0]
assert X.shape[1] == 8

# Standardize inputs for smoother training.
X_mean = X.mean(axis=0, keepdims=True)
X_std = X.std(axis=0, keepdims=True) + 1e-8
X = (X - X_mean) / X_std

# Standardize targets for easier optimization.
y_mean = y.mean(axis=0, keepdims=True)
y_std = y.std(axis=0, keepdims=True) + 1e-8
y = (y - y_mean) / y_std

# Split into train and validation sets.
idx = np.arange(X.shape[0])
np.random.shuffle(idx)
cut = int(0.8 * len(idx))

# Apply the shuffled split indices.
train_idx = idx[:cut]
val_idx = idx[cut:]
X_train = X[train_idx]
X_val = X[val_idx]

y_train = y[train_idx]
y_val = y[val_idx]

# Build small initial network parameters.
input_dim = X_train.shape[1]
hidden_dim = 10
output_dim = 1

# Create a helper for initialization.
def init_params():
    W1 = 0.1 * np.random.randn(input_dim, hidden_dim)
    b1 = np.zeros((1, hidden_dim))
    W2 = 0.1 * np.random.randn(hidden_dim, output_dim)
    b2 = np.zeros((1, output_dim))

    return [W1, b1, W2, b2]

# Define the forward pass.
def forward_pass(X_batch, params):
    W1, b1, W2, b2 = params
    Z1 = X_batch @ W1 + b1
    A1 = np.maximum(0.0, Z1)
    Yhat = A1 @ W2 + b2

    return Z1, A1, Yhat

# Define mean squared error loss.
def mse_loss(y_true, y_pred):
    diff = y_pred - y_true
    return np.mean(diff * diff)

# Define backpropagation gradients.
def backward_pass(X_batch, y_true, params):
    W1, b1, W2, b2 = params
    Z1, A1, Yhat = forward_pass(X_batch, params)
    n = X_batch.shape[0]

    dY = (2.0 / n) * (Yhat - y_true)
    dW2 = A1.T @ dY
    db2 = np.sum(dY, axis=0, keepdims=True)
    dA1 = dY @ W2.T

    dZ1 = dA1 * (Z1 > 0)
    dW1 = X_batch.T @ dZ1
    db1 = np.sum(dZ1, axis=0, keepdims=True)

    return [dW1, db1, dW2, db2]

# Define plain gradient descent training.
def train_gd(X_data, y_data, epochs, lr):
    params = init_params()
    history = []

    for epoch in range(epochs):
        grads = backward_pass(X_data, y_data, params)
        step = 0

        for p, g in zip(params, grads):
            params[step] = p - lr * g
            step = step + 1

        _, _, pred = forward_pass(X_data, params)
        history.append(mse_loss(y_data, pred))

    return params, history

# Define Adam optimizer training.
def train_adam(X_data, y_data, epochs, lr):
    params = init_params()
    history = []
    m = []
    v = []

    for p in params:
        m.append(np.zeros_like(p))
        v.append(np.zeros_like(p))

    beta1 = 0.9
    beta2 = 0.999
    eps = 1e-8

    for epoch in range(epochs):
        grads = backward_pass(X_data, y_data, params)
        t = epoch + 1
        step = 0

        for p, g in zip(params, grads):
            m[step] = beta1 * m[step] + (1 - beta1) * g
            v[step] = beta2 * v[step] + (1 - beta2) * (g * g)
            m_hat = m[step] / (1 - beta1 ** t)

            v_hat = v[step] / (1 - beta2 ** t)
            params[step] = p - lr * m_hat / (np.sqrt(v_hat) + eps)
            step = step + 1

        _, _, pred = forward_pass(X_data, params)
        history.append(mse_loss(y_data, pred))

    return params, history

# Train both optimizers on identical data.
epochs = 200
gd_params, gd_history = train_gd(X_train, y_train, epochs, 0.03)
adam_params, adam_history = train_adam(X_train, y_train, epochs, 0.01)

# Evaluate validation losses after training.
_, _, gd_val_pred = forward_pass(X_val, gd_params)
_, _, adam_val_pred = forward_pass(X_val, adam_params)

gd_val_loss = mse_loss(y_val, gd_val_pred)
adam_val_loss = mse_loss(y_val, adam_val_pred)

# Print a short teaching summary.
print("Rows used:", X.shape[0], "Features:", X.shape[1])
print("GD first loss:", round(gd_history[0], 4))
print("GD final loss:", round(gd_history[-1], 4))
print("Adam first loss:", round(adam_history[0], 4))
print("Adam final loss:", round(adam_history[-1], 4))
print("GD validation loss:", round(gd_val_loss, 4))
print("Adam validation loss:", round(adam_val_loss, 4))
print("Adam usually drops faster early in training.")



Rows used: 300 Features: 8
GD first loss: 0.949
GD final loss: 0.2868
Adam first loss: 0.9574
Adam final loss: 0.0942
GD validation loss: 0.2986
Adam validation loss: 0.1436
Adam usually drops faster early in training.


### **3.3. Optimization Strategy Comparison**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/JHU/Artificial Intelligence in Civil Engineering Applications/Module_05/Lecture_B/image_03_03.jpg?v=1774623537" width="250">



>* Optimizers affect learning speed and stability.
>* SGD is faster but noisier than GD.

>* Momentum smooths updates and speeds training.
>* Adam adapts steps, but may generalize worse.

>* Compare training dynamics and final performance.
>* Optimizer choice depends on tuning and context.



In [9]:
#@title Python Code - Optimization Strategy Comparison

# Compare optimizers for a simple neural network.
# Use air quality data for regression.
# Focus on convergence, stability, and interpretation.

# Download commands are shown for Colab.
# !wget -O AirQualityUCI.zip https://archive.ics.uci.edu/ml/machine-learning-databases/00360/AirQualityUCI.zip
# !unzip -o AirQualityUCI.zip

import os
import zipfile
import numpy as np
import pandas as pd

# Set deterministic seeds for repeatability.
np.random.seed(7)
zip_path = "AirQualityUCI.zip"
csv_path = "AirQualityUCI.csv"
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00360/AirQualityUCI.zip"

# Download the dataset if missing.
if not os.path.exists(zip_path):
    import urllib.request
    urllib.request.urlretrieve(url, zip_path)

# Extract the csv file safely.
if not os.path.exists(csv_path):
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(".")

# Load the semicolon separated dataset.
df = pd.read_csv(csv_path, sep=";", decimal=",")
df = df.dropna(axis=1, how="all")
df = df.dropna(axis=0, how="all")

# Replace missing markers with NaN.
df = df.replace(-200, np.nan)
feature_cols = ["CO(GT)", "PT08.S1(CO)", "NMHC(GT)", "C6H6(GT)"]
target_col = "NO2(GT)"

# Keep only needed columns.
df = df[feature_cols + [target_col]].copy()
df = df.dropna()
if len(df) < 300:
    raise ValueError("Dataset became too small after cleaning.")

# Use a small subset for fast teaching.
df = df.iloc[:1200].copy()
X = df[feature_cols].to_numpy(dtype=float)
y = df[[target_col]].to_numpy(dtype=float)

# Split into train and validation sets.
cut = int(0.8 * len(X))
X_train = X[:cut]
y_train = y[:cut]
X_val = X[cut:]
y_val = y[cut:]

# Standardize features using training statistics.
x_mean = X_train.mean(axis=0, keepdims=True)
x_std = X_train.std(axis=0, keepdims=True) + 1e-8
X_train = (X_train - x_mean) / x_std
X_val = (X_val - x_mean) / x_std

# Standardize target for stable training.
y_mean = y_train.mean(axis=0, keepdims=True)
y_std = y_train.std(axis=0, keepdims=True) + 1e-8
y_train_s = (y_train - y_mean) / y_std
y_val_s = (y_val - y_mean) / y_std

# Check shapes before training.
if X_train.shape[1] != 4:
    raise ValueError("Expected four input features.")
if y_train_s.shape[1] != 1:
    raise ValueError("Expected one target column.")

# Build a tiny neural network.
def init_params(input_dim, hidden_dim):
    W1 = 0.1 * np.random.randn(input_dim, hidden_dim)
    b1 = np.zeros((1, hidden_dim))
    W2 = 0.1 * np.random.randn(hidden_dim, 1)
    b2 = np.zeros((1, 1))
    return [W1, b1, W2, b2]

# Run forward propagation.
def forward_pass(Xb, params):
    W1, b1, W2, b2 = params
    Z1 = Xb.dot(W1) + b1
    A1 = np.tanh(Z1)
    Yh = A1.dot(W2) + b2
    return Z1, A1, Yh

# Compute mean squared error.
def mse_loss(y_true, y_pred):
    diff = y_pred - y_true
    return np.mean(diff * diff)

# Run backward propagation.
def backward_pass(Xb, yb, params, cache):
    Z1, A1, Yh = cache
    W1, b1, W2, b2 = params
    m = Xb.shape[0]
    dY = 2.0 * (Yh - yb) / m
    dW2 = A1.T.dot(dY)
    db2 = np.sum(dY, axis=0, keepdims=True)
    dA1 = dY.dot(W2.T)
    dZ1 = dA1 * (1.0 - np.tanh(Z1) ** 2)
    dW1 = Xb.T.dot(dZ1)
    db1 = np.sum(dZ1, axis=0, keepdims=True)
    return [dW1, db1, dW2, db2]

# Evaluate losses on both splits.
def evaluate_losses(params):
    train_pred = forward_pass(X_train, params)[2]
    val_pred = forward_pass(X_val, params)[2]
    train_loss = mse_loss(y_train_s, train_pred)
    val_loss = mse_loss(y_val_s, val_pred)
    return train_loss, val_loss

# Train with selected optimizer.
def train_model(name, lr, epochs, batch_size, momentum):
    params = init_params(X_train.shape[1], 8)
    v = [np.zeros_like(p) for p in params]
    m1 = [np.zeros_like(p) for p in params]
    m2 = [np.zeros_like(p) for p in params]
    beta1 = 0.9
    beta2 = 0.999
    eps = 1e-8
    history = []
    n = X_train.shape[0]

    for epoch in range(epochs):
        order = np.arange(n)
        np.random.shuffle(order)
        Xs = X_train[order]
        ys = y_train_s[order]

        for start in range(0, n, batch_size):
            end = min(start + batch_size, n)
            Xb = Xs[start:end]
            yb = ys[start:end]
            cache = forward_pass(Xb, params)
            grads = backward_pass(Xb, yb, params, cache)

            if name == "batch_gd":
                for i in range(4):
                    params[i] = params[i] - lr * grads[i]

            elif name == "sgd_momentum":
                for i in range(4):
                    v[i] = momentum * v[i] - lr * grads[i]
                    params[i] = params[i] + v[i]

            elif name == "adam":
                step = epoch * ((n + batch_size - 1) // batch_size)
                step = step + (start // batch_size) + 1
                for i in range(4):
                    m1[i] = beta1 * m1[i] + (1 - beta1) * grads[i]
                    m2[i] = beta2 * m2[i] + (1 - beta2) * (grads[i] * grads[i])
                    m1h = m1[i] / (1 - beta1 ** step)
                    m2h = m2[i] / (1 - beta2 ** step)
                    params[i] = params[i] - lr * m1h / (np.sqrt(m2h) + eps)

        train_loss, val_loss = evaluate_losses(params)
        history.append([train_loss, val_loss])

    return params, np.array(history)

# Train three optimization strategies.
params_gd, hist_gd = train_model("batch_gd", 0.03, 40, len(X_train), 0.0)
params_sgd, hist_sgd = train_model("sgd_momentum", 0.02, 40, 32, 0.9)
params_adam, hist_adam = train_model("adam", 0.01, 40, 32, 0.0)

# Summarize convergence and stability.
def summarize(name, hist):
    first = float(hist[0, 1])
    last = float(hist[-1, 1])
    changes = np.abs(np.diff(hist[:, 1]))
    stability = float(np.mean(changes))
    return [name, round(first, 4), round(last, 4), round(stability, 4)]

# Create a compact comparison table.
rows = []
rows.append(summarize("Full-batch GD", hist_gd))
rows.append(summarize("Mini-batch SGD", hist_sgd))
rows.append(summarize("Adam", hist_adam))
result = pd.DataFrame(rows, columns=["Method", "Start val loss", "End val loss", "Avg change"])

# Print a short beginner-friendly interpretation.
print("Air quality target: predict NO2 from four sensor features.")
print("Smaller end loss is better; smaller avg change is steadier.")
print(result.to_string(index=False))

# Pick the lowest final validation loss.
best_idx = int(np.argmin(result["End val loss"].to_numpy()))
best_name = result.iloc[best_idx, 0]
print("Best final validation loss:", best_name)



Air quality target: predict NO2 from four sensor features.
Smaller end loss is better; smaller avg change is steadier.
        Method  Start val loss  End val loss  Avg change
 Full-batch GD          1.1929        0.2643      0.0238
Mini-batch SGD          0.2987        0.2450      0.0165
          Adam          0.2616        0.2445      0.0169
Best final validation loss: Adam


# <font color="#418FDE" size="6.5" uppercase>**Neural Network Optimization and Training**</font>


In this lecture, you learned to:
- Formulate neural network training as an optimization problem involving loss functions, gradients, and parameter updates. 
- Implement gradient-based training methods, including gradient descent, stochastic gradient descent, and backpropagation. 
- Evaluate the effects of optimization choices such as learning rate, momentum, batch size, and Adam on neural network. 

In the next Module (Module 6), we will go over 'Neural Networks, Part II'